# RivalRadar Agentic Pipeline (Agent 1 + Agent 2)

This notebook combines:
- Agent 1: competitor page collection + structured parsing
- Agent 2: competitive positioning analysis (vulnerability scoring)

Goal: run a single pipeline from scrape/structured input to ranked risk outputs.

## 1. Imports and Shared Configuration

In [37]:
import asyncio
import importlib
import json
import time
from pathlib import Path
from typing import Any

import pandas as pd

from competitor_targets import COMPETITOR_TARGETS
from scrapers.output_parser import parse_pricing_page, save_structured

import agents.agent2_analyzer as analyzer_module
analyzer_module = importlib.reload(analyzer_module)
analyze_profiles = analyzer_module.analyze_profiles
print_vulnerability_report = analyzer_module.print_vulnerability_report
save_vulnerability_report = analyzer_module.save_vulnerability_report

import agents.agent3_pricing as pricing_module
pricing_module = importlib.reload(pricing_module)
predict_pricing_changes = pricing_module.predict_pricing_changes
print_pricing_predictions = pricing_module.print_pricing_predictions
save_pricing_predictions = pricing_module.save_pricing_predictions

import agents.agent4_actions as actions_module
actions_module = importlib.reload(actions_module)
generate_action_recommendations = actions_module.generate_action_recommendations
print_action_recommendations = actions_module.print_action_recommendations
save_action_recommendations = actions_module.save_action_recommendations

# Crawlee is optional for live scraping; fallback mode can still run without it.
try:
    from scrapers.crawlee_scraper import CrawleeScraper
    CRAWLEE_AVAILABLE = True
except ModuleNotFoundError:
    CrawleeScraper = None
    CRAWLEE_AVAILABLE = False

OUTPUT_DIR = Path("scraper_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_LIVE_SCRAPE = CRAWLEE_AVAILABLE
VERBOSE = True
MAX_RETRIES = 1

print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Competitors configured: {len(COMPETITOR_TARGETS)}")
print(f"Crawlee available: {CRAWLEE_AVAILABLE}")
print(f"Live scrape mode: {RUN_LIVE_SCRAPE}")

if not CRAWLEE_AVAILABLE:
    print("Install missing deps in your venv: pip install -r requirements.txt")

Output dir: \\wsl.localhost\Ubuntu\home\swathika\founders5115\DBA5115-rivalradar\rivalradar\scraper_output
Competitors configured: 5
Crawlee available: False
Live scrape mode: False
Install missing deps in your venv: pip install -r requirements.txt


## 2. Define First Agent Interface and Implementation

In [38]:
class Agent1Collector:
    """Collect competitor pricing pages and emit structured competitor profiles."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.scraper = CrawleeScraper() if CRAWLEE_AVAILABLE else None

    def build_targets(self) -> list[dict[str, str]]:
        return [
            {
                "url": target["pricing_url"],
                "page_type": "pricing",
                "name": target["name"],
            }
            for target in COMPETITOR_TARGETS
        ]

    async def collect(self) -> dict[str, Any]:
        if self.scraper is None:
            raise ModuleNotFoundError(
                "crawlee is not installed. Install deps with: pip install -r requirements.txt"
            )

        targets = self.build_targets()
        scrape_results = await self.scraper.scrape_pages(targets)

        profiles: list[dict[str, Any]] = []
        for result in scrape_results:
            profile = parse_pricing_page(
                raw_text=result.get("raw_text", ""),
                company=result.get("name", "Unknown"),
                url=result.get("url", ""),
                scraped_at=result.get("scraped_at", ""),
            )
            save_structured(profile, self.output_dir)
            profiles.append(profile)

        return {
            "agent": "agent1_collector",
            "target_count": len(targets),
            "scrape_result_count": len(scrape_results),
            "structured_profiles": profiles,
        }


def load_existing_structured_profiles(output_dir: Path) -> list[dict[str, Any]]:
    profiles: list[dict[str, Any]] = []
    for path in sorted(output_dir.glob("*_structured.json")):
        with open(path, encoding="utf-8") as f:
            profiles.append(json.load(f))
    return profiles

## 3. Define Second Agent Using First Agent Output

In [39]:
class Agent2CompetitiveAnalyzer:
    """Analyze competitor profiles and return vulnerability-focused outputs."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def analyze(self, agent1_output: dict[str, Any]) -> dict[str, Any]:
        profiles = agent1_output["structured_profiles"]
        vulnerability_results = analyze_profiles(profiles)
        report_path = save_vulnerability_report(vulnerability_results, self.output_dir)

        return {
            "agent": "agent2_analyzer",
            "vulnerability_results": vulnerability_results,
            "report_path": str(report_path),
            "analyzed_count": len(vulnerability_results),
        }

In [40]:
class Agent3PricingPredictor:
    """Predict competitor pricing-change likelihood and timeline."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def predict(self, agent1_output: dict[str, Any], agent2_output: dict[str, Any]) -> dict[str, Any]:
        predictions = predict_pricing_changes(
            agent2_output["vulnerability_results"],
            agent1_output["structured_profiles"],
        )
        report_path = save_pricing_predictions(predictions, self.output_dir)
        return {
            "agent": "agent3_pricing",
            "pricing_predictions": predictions,
            "report_path": str(report_path),
            "prediction_count": len(predictions),
        }


class Agent4ActionPlanner:
    """Generate VC-facing action recommendations from Agent 2 + Agent 3 outputs."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def recommend(self, agent2_output: dict[str, Any], agent3_output: dict[str, Any]) -> dict[str, Any]:
        recommendations = generate_action_recommendations(
            agent2_output["vulnerability_results"],
            agent3_output["pricing_predictions"],
        )
        report_path = save_action_recommendations(recommendations, self.output_dir)
        return {
            "agent": "agent4_actions",
            "action_recommendations": recommendations,
            "report_path": str(report_path),
            "recommendation_count": len(recommendations),
        }

## 4. Define Third and Fourth Agents (Pricing + Actions)

## 5. Compose Multi-Agent Pipeline (Agents 1-4)

In [41]:
async def run_agent_pipeline(run_live_scrape: bool = True) -> dict[str, Any]:
    start = time.perf_counter()

    agent1 = Agent1Collector(OUTPUT_DIR)
    agent2 = Agent2CompetitiveAnalyzer(OUTPUT_DIR)
    agent3 = Agent3PricingPredictor(OUTPUT_DIR)
    agent4 = Agent4ActionPlanner(OUTPUT_DIR)

    agent1_output: dict[str, Any] | None = None

    if run_live_scrape:
        for attempt in range(MAX_RETRIES + 1):
            try:
                if VERBOSE:
                    print(f"[Agent1] Live scrape attempt {attempt + 1}/{MAX_RETRIES + 1}")
                agent1_output = await agent1.collect()
                break
            except Exception as exc:
                print(f"[Agent1] Attempt failed: {exc}")
                if attempt == MAX_RETRIES:
                    print("[Agent1] Falling back to existing structured JSON files.")

    if agent1_output is None:
        profiles = load_existing_structured_profiles(OUTPUT_DIR)
        agent1_output = {
            "agent": "agent1_collector_fallback",
            "target_count": len(COMPETITOR_TARGETS),
            "scrape_result_count": 0,
            "structured_profiles": profiles,
        }

    agent2_output = agent2.analyze(agent1_output)
    agent3_output = agent3.predict(agent1_output, agent2_output)
    agent4_output = agent4.recommend(agent2_output, agent3_output)

    elapsed = time.perf_counter() - start
    if VERBOSE:
        print(f"[Pipeline] Completed in {elapsed:.2f}s")

    return {
        "elapsed_seconds": elapsed,
        "agent1_output": agent1_output,
        "agent2_output": agent2_output,
        "agent3_output": agent3_output,
        "agent4_output": agent4_output,
    }

## 6. Run End-to-End Example (Agents 1-4)

In [42]:
pipeline_result = await run_agent_pipeline(run_live_scrape=RUN_LIVE_SCRAPE)

print("Agent 1 output keys:", list(pipeline_result["agent1_output"].keys()))
print("Agent 2 output keys:", list(pipeline_result["agent2_output"].keys()))
print("Agent 3 output keys:", list(pipeline_result["agent3_output"].keys()))
print("Agent 4 output keys:", list(pipeline_result["agent4_output"].keys()))

vulnerability_results = pipeline_result["agent2_output"]["vulnerability_results"]
pricing_predictions = pipeline_result["agent3_output"]["pricing_predictions"]
action_recommendations = pipeline_result["agent4_output"]["action_recommendations"]

print_vulnerability_report(vulnerability_results)
print_pricing_predictions(pricing_predictions)
print_action_recommendations(action_recommendations)

print("Agent 2 report:", pipeline_result["agent2_output"]["report_path"])
print("Agent 3 report:", pipeline_result["agent3_output"]["report_path"])
print("Agent 4 report:", pipeline_result["agent4_output"]["report_path"])

[Pipeline] Completed in 0.05s
Agent 1 output keys: ['agent', 'target_count', 'scrape_result_count', 'structured_profiles']
Agent 2 output keys: ['agent', 'vulnerability_results', 'report_path', 'analyzed_count']
Agent 3 output keys: ['agent', 'pricing_predictions', 'report_path', 'prediction_count']
Agent 4 output keys: ['agent', 'action_recommendations', 'report_path', 'recommendation_count']

                        COMPETITIVE VULNERABILITY ANALYSIS (EXPLAINABLE)                        

[ClickUp] score=0.965 risk=CRITICAL confidence=0.919 rank=1/5
Summary: Ranked #1/5 (peer_percentile=1.000). Top driver: pricing_pressure (0.280). Overall vulnerability=0.894 (critical).
Component breakdown:
  - pricing_pressure         score=1.000 weight=0.28 contribution=0.280
    reasoning: free tier present, low entry point ($0.00), wide spread ($28.00)
  - segment_coverage         score=1.000 weight=0.22 contribution=0.220
    reasoning: 5 plans, enterprise tier, free tier
  - feature_depth     

In [43]:
vulnerability_rows = []
for result in vulnerability_results:
    top_component = max(result.component_breakdown, key=lambda c: c.weighted_score)
    vulnerability_rows.append(
        {
            "company": result.company,
            "rank": result.peer_rank,
            "vulnerability_score": result.vulnerability_score,
            "risk_level": result.risk_level,
            "confidence": result.confidence,
            "top_driver": top_component.name,
            "signal_hits": sum(result.signals.values()),
            "reasoning_summary": result.reasoning_summary,
        }
    )

pricing_rows = [
    {
        "company": p.company,
        "change_probability": p.change_probability,
        "pricing_risk": p.risk_level,
        "predicted_timeline": p.predicted_timeline,
        "top_driver": max(p.drivers.items(), key=lambda item: item[1])[0],
        "reasoning_summary": p.reasoning_summary,
    }
    for p in pricing_predictions
]

action_rows = [
    {
        "company": a.company,
        "priority": a.priority,
        "recommendation_type": a.recommendation_type,
        "owner": a.owner,
        "due_window": a.due_window,
        "action_title": a.action_title,
        "rationale": a.rationale,
    }
    for a in action_recommendations
]

vulnerability_df = pd.DataFrame(vulnerability_rows).sort_values("rank")
pricing_df = pd.DataFrame(pricing_rows).sort_values("change_probability", ascending=False)
actions_df = pd.DataFrame(action_rows).sort_values("priority")

pd.set_option("display.max_colwidth", 220)
print("VULNERABILITY SUMMARY")
display(vulnerability_df)
print("PRICING PREDICTION SUMMARY")
display(pricing_df)
print("ACTION RECOMMENDATION SUMMARY")
display(actions_df)

VULNERABILITY SUMMARY


,company,rank,vulnerability_score,risk_level,confidence,top_driver,signal_hits,reasoning_summary
0,ClickUp,1,0.965,critical,0.919,pricing_pressure,13,Ranked #1/5 (peer_percentile=1.000). Top driver: pricing_pressure (0.280). Overall vulnerability=0.894 (critical).
1,Notion,2,0.891,critical,0.811,pricing_pressure,7,Ranked #2/5 (peer_percentile=0.750). Top driver: pricing_pressure (0.244). Overall vulnerability=0.860 (critical).
2,Linear,3,0.703,high,0.782,pricing_pressure,6,Ranked #3/5 (peer_percentile=0.500). Top driver: pricing_pressure (0.244). Overall vulnerability=0.692 (high).
3,HubSpot,4,0.655,high,0.750,segment_coverage,3,Ranked #4/5 (peer_percentile=0.250). Top driver: segment_coverage (0.220). Overall vulnerability=0.628 (medium).
4,Stripe,5,0.040,low,0.400,business_model_pressure,0,Ranked #5/5 (peer_percentile=0.000). Top driver: business_model_pressure (0.120). Overall vulnerability=0.120 (low).


PRICING PREDICTION SUMMARY


,company,change_probability,pricing_risk,predicted_timeline,top_driver,reasoning_summary
0,ClickUp,0.910,high,0-30 days,strategic_signal_intensity,ClickUp pricing-change probability=0.910 (high) in 0-30 days. Top driver: strategic_signal_intensity.
1,Notion,0.762,medium,1-2 months,vulnerability_pressure,Notion pricing-change probability=0.762 (medium) in 1-2 months. Top driver: vulnerability_pressure.
2,Linear,0.688,medium,1-2 months,plan_complexity,Linear pricing-change probability=0.688 (medium) in 1-2 months. Top driver: plan_complexity.
3,HubSpot,0.588,medium,2-3 months,plan_complexity,HubSpot pricing-change probability=0.588 (medium) in 2-3 months. Top driver: plan_complexity.
4,Stripe,0.182,low,3+ months,portfolio_competition_intensity,Stripe pricing-change probability=0.182 (low) in 3+ months. Top driver: portfolio_competition_intensity.


ACTION RECOMMENDATION SUMMARY


,company,priority,recommendation_type,owner,due_window,action_title,rationale
0,ClickUp,P0,pricing_response,Portfolio Operations Lead,48 hours,Run pricing defense sprint against ClickUp,ClickUp is critical risk with vulnerability score 0.965; predicted pricing change probability is 0.910 (0-30 days).
1,Notion,P0,product_response,Product Advisor / CTO Partner,48 hours,Launch feature-gap closure plan for Notion,Notion is critical risk with vulnerability score 0.891; predicted pricing change probability is 0.762 (1-2 months).
2,Linear,P1,product_response,Product Advisor / CTO Partner,7 days,Launch feature-gap closure plan for Linear,Linear is high risk with vulnerability score 0.703; predicted pricing change probability is 0.688 (1-2 months).
3,HubSpot,P2,gtm_response,GTM Advisor,14 days,Prepare GTM counter-messaging against HubSpot,HubSpot is high risk with vulnerability score 0.655; predicted pricing change probability is 0.588 (2-3 months).
4,Stripe,P3,monitor_only,Analyst,30 days,Continue structured monitoring for Stripe,Stripe is low risk with vulnerability score 0.040; predicted pricing change probability is 0.182 (3+ months).


## 7. Validation and Debug Output (Agents 1-4)

In [44]:
agent1_output = pipeline_result["agent1_output"]
agent2_output = pipeline_result["agent2_output"]
agent3_output = pipeline_result["agent3_output"]
agent4_output = pipeline_result["agent4_output"]

assert "structured_profiles" in agent1_output, "Agent 1 output missing structured_profiles"
assert isinstance(agent1_output["structured_profiles"], list), "structured_profiles must be a list"
assert len(agent1_output["structured_profiles"]) > 0, "No structured profiles available"

assert "vulnerability_results" in agent2_output, "Agent 2 output missing vulnerability_results"
assert isinstance(agent2_output["vulnerability_results"], list), "vulnerability_results must be a list"
assert len(agent2_output["vulnerability_results"]) > 0, "Agent 2 returned no vulnerability results"

assert "pricing_predictions" in agent3_output, "Agent 3 output missing pricing_predictions"
assert isinstance(agent3_output["pricing_predictions"], list), "pricing_predictions must be a list"
assert len(agent3_output["pricing_predictions"]) > 0, "Agent 3 returned no pricing predictions"

assert "action_recommendations" in agent4_output, "Agent 4 output missing action_recommendations"
assert isinstance(agent4_output["action_recommendations"], list), "action_recommendations must be a list"
assert len(agent4_output["action_recommendations"]) > 0, "Agent 4 returned no recommendations"

sample_profile = agent1_output["structured_profiles"][0]
required_profile_keys = {"company", "pricing_summary", "raw_char_count"}
assert required_profile_keys.issubset(sample_profile.keys()), "Profile schema mismatch"

if VERBOSE:
    print("[Validation] Agents 1-4 outputs passed schema checks.")
    print("[Debug] Structured profiles:", len(agent1_output["structured_profiles"]))
    print("[Debug] Vulnerability results:", len(agent2_output["vulnerability_results"]))
    print("[Debug] Pricing predictions:", len(agent3_output["pricing_predictions"]))
    print("[Debug] Action recommendations:", len(agent4_output["action_recommendations"]))
    print("[Debug] Pipeline elapsed seconds:", round(pipeline_result["elapsed_seconds"], 2))

[Validation] Agents 1-4 outputs passed schema checks.
[Debug] Structured profiles: 5
[Debug] Vulnerability results: 5
[Debug] Pricing predictions: 5
[Debug] Action recommendations: 5
[Debug] Pipeline elapsed seconds: 0.05
